# 多重振り子の解法

In [ ]:
!pip install jax

In [ ]:
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp


#

# Implement of Lagrangian



$$
K = \frac{1}{2}m_1v_1^2+ \frac{1}{2}m_2v_2^2
$$
$$
V = -m_1 g l_1\cos\theta_1 +  m_2 g (l_1\cos\theta_1+l_2\cos\theta_2)
$$
$$
L= K - V
$$



$$
x_1 = l_1 \sin \theta_1, \quad y_1 = l_1 \cos \theta_2
$$

$$
x_2 = l_1 \sin \theta_1+ l_2 \sin \theta_2, \quad y_2 = l_1 \cos \theta_1 + l_2 \cos \theta_2
$$

$$
v_1^2 = \dot{x_1}^2+ \dot{y_1}^2 = l_1^2 \dot{\theta_1}^2
$$

$$
v_2^2 = \dot{x_2}^2+ \dot{y_2}^2 =  l_1^2 \dot{\theta_2}^2+ l_2^2 \dot{\theta_2}^2+ 2l_1l_2\dot{\theta_1}\dot{\theta_2} \cos(\theta_1- \theta_2)
$$

## note 1

$$
x = x(t)
$$
$$
y = f(x)
$$

$$
\dot{y} = \frac{d}{dt}y = \frac{d}{dx}\frac{dx}{dt} y =\dot{x} \frac{dy}{dx}
$$

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jacfwd
from jax.experimental.ode import odeint

# パラメータ
m1, m2 = 1.0, 1.0
l1, l2 = 1.0, 1.0
g = 9.81

# ラグランジアン
def lagrangian(q, qdot):
    theta1, theta2 = q
    dtheta1, dtheta2 = qdot

    # 位置
    x1 = l1 * jnp.sin(theta1)
    y1 = -l1 * jnp.cos(theta1)

    x2 = x1 + l2 * jnp.sin(theta2)
    y2 = y1 - l2 * jnp.cos(theta2)

    # 速度（ヤコビアンで計算）
    def pos(q):
        t1, t2 = q
        x1 = l1 * jnp.sin(t1)
        y1 = -l1 * jnp.cos(t1)
        x2 = x1 + l2 * jnp.sin(t2)
        y2 = y1 - l2 * jnp.cos(t2)
        return jnp.array([x1, y1, x2, y2])

    J = jacfwd(pos)(q)  # 4x2
    v = J @ qdot        # 速度

    v1 = v[:2]
    v2 = v[2:]

    # エネルギー
    T = 0.5*m1*jnp.dot(v1, v1) + 0.5*m2*jnp.dot(v2, v2)
    V = m1*g*y1 + m2*g*y2

    return T - V

# Solve EOM

以下、$q$, $\dot{q}$はベクトル、$q = (\theta_1, \theta_2)$,  $\dot{q} = (\dot{\theta_1},\dot{\theta_2})$を表すとする。
$$
\dot{q}, \ddot{q} = f(t, \mathrm{state})
$$

$$
f(t, \mathrm{state}) = M^{-1}(q)\left(\frac{\partial L}{\partial q} - C(q, \dot{q})\right)
$$

1. lagrangian を`0`成分(角度)で微分
```python
    dL_dq = grad(lagrangian, 0)(q, qdot)
```
2. lagrangian を`1`成分(角速度)で微分
```python
    dL_dqdot = grad(lagrangian, 1)(q, qdot)
```

3.
$$
\frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}}\right)  = \frac{\partial^2 L}{\partial q \partial \dot{q}} \dot{q}+\frac{\partial^2 L}{\partial \dot{q}^2}\ddot{q}
$$

$\frac{\partial^2 L}{\partial \dot{q}^2}\ddot{q}$の項はいったん0としてスルー（最終的にこの部分が欲しいので）
```python
    ddt_term = jacfwd(dL_dqdot_fn, (0,1))(q, qdot)
    ddt = ddt_term[0] @ qdot + ddt_term[1] @ jnp.zeros_like(qdot)
```

4.

運動方程式の質量に相当する部分を求める。
さらに、ラグランジアンの右辺を整理する。
つまり以下の式の右辺を完成させている。
$$
\frac{\partial^2 L}{\partial \dot{q}^2}\ddot{q} = \frac{\partial L}{\partial q}- \frac{\partial^2 L}{\partial q \partial \dot{q}} \dot{q} \quad(*)
$$
```python
    M = jacfwd(grad(lagrangian, 1), 1)(q, qdot)
    rhs = dL_dq - ddt
```

5.
np.linarg.solveで各速度を求める。
式(*)は結局
$$
A \boldsymbol{x} = \boldsymbol{b}
$$
の事である。
実際に、

$A= frac{\partial^2 L}{\partial \dot{q}^2}$,

$b =\frac{\partial L}{\partial q}- \frac{\partial^2 L}{\partial q \partial \dot{q}} \dot{q} $

$\boldsymbol{x} = \ddot{q}$

と対応している。

In [ ]:

def eom(state, t):
    q = state[:2]
    qdot = state[2:]

    # ∂L/∂q
    dL_dq = grad(lagrangian, 0)(q, qdot)

    # ∂L/∂qdot
    dL_dqdot = grad(lagrangian, 1)(q, qdot)

    # 時間微分 d/dt (∂L/∂qdot)
    def dL_dqdot_fn(q, qdot):
        return grad(lagrangian, 1)(q, qdot)

    ddt_term = jacfwd(dL_dqdot_fn, (0,1))(q, qdot)
    ddt = ddt_term[0] @ qdot + ddt_term[1] @ jnp.zeros_like(qdot)

    # M(q) = ∂²L/∂qdot²
    M = jacfwd(grad(lagrangian, 1), 1)(q, qdot)

    # 右辺
    rhs = dL_dq - ddt

    # 加速度
    qddot = jnp.linalg.solve(M, rhs)

    return jnp.concatenate([qdot, qddot])

# `odeint`について

jax.experimental.ode.odeintやscipy.integrate.odeintのodeint関数は以下の常微分方程式を解いている
$$
\frac{d \boldsymbol{x}}{d t} = f(\boldsymbol{x})
$$
今回は
$$
\frac{d }{d t}\dot{q} = f(q, \dot{q})
$$

このように返り値を配列にして解けば、多階の常微分方程式や連立常微分方程式が簡単に解ける。
```python
def f(xv, t):
    return np.array([xv[1], -xv[1] - xv[0]])

xv0 = np.array([0, 1])
t = np.arange(0, 10.1, 0.1)
xv = odeint(f, xv0, t)
```



In [ ]:
t = jnp.linspace(0, 100, 2000)

# 初期条件
state0 = jnp.array([
    jnp.pi/2,   # theta1
    jnp.pi/2,   # theta2
    0.0,        # dtheta1
    0.0         # dtheta2
])

traj = odeint(eom, state0, t)

In [ ]:
import matplotlib.pyplot as plt
def get_positions(q):
    theta1, theta2 = q

    x1 = l1 * jnp.sin(theta1)
    y1 = -l1 * jnp.cos(theta1)

    x2 = x1 + l2 * jnp.sin(theta2)
    y2 = y1 - l2 * jnp.cos(theta2)

    return jnp.array([
        [x1, y1],  # 質点1
        [x2, y2]   # 質点2
    ])

In [ ]:
# 角度だけ取り出し
qs = traj[:, :2]

# vmapで一気に座標化
positions = jax.vmap(get_positions)(qs)
x1 = positions[:, 0, 0]
y1 = positions[:, 0, 1]

x2 = positions[:, 1, 0]
y2 = positions[:, 1, 1]

In [ ]:

import matplotlib.pyplot as plt
plt.plot(x1, y1)
plt.plot(x2, y2)
plt.gca().set_aspect('equal')
plt.title("Double Pendulum Trajectory")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

x1 = np.array(x1)
y1 = np.array(y1)
x2 = np.array(x2)
y2 = np.array(y2)

fig, ax = plt.subplots()
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')

# 棒
line, = ax.plot([], [], 'o-', lw=2)

# 質点（最初に作る）
mass1, = ax.plot([], [], 'o', color='red')
mass2, = ax.plot([], [], 'o', color='blue')

# 軌跡
trail1, = ax.plot([], [], '--', alpha=0.6, color='red')
trail2, = ax.plot([], [], '--', alpha=0.6, color='blue')

def init():
    line.set_data([], [])
    mass1.set_data([], [])
    mass2.set_data([], [])
    trail1.set_data([], [])
    trail2.set_data([], [])
    return line, mass1, mass2, trail1, trail2

def update(i):
    xs = [0, x1[i], x2[i]]
    ys = [0, y1[i], y2[i]]

    line.set_data(xs, ys)

    # 修正ポイント
    mass1.set_data([x1[i]], [y1[i]])
    mass2.set_data([x2[i]], [y2[i]])

    trail1.set_data(x1[:i], y1[:i])
    trail2.set_data(x2[:i], y2[:i])

    return line, mass1, mass2, trail1, trail2

ani = FuncAnimation(
    fig, update,
    frames=len(x1),
    init_func=init,
    interval=20,
    blit=True
)

from IPython.display import HTML
HTML(ani.to_jshtml())
